In [2]:
!pip install torch torchvision timm

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm


In [2]:
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])


In [3]:
train_data = datasets.ImageFolder(r"C:\Users\sanjk\OneDrive\Desktop\rosedata", transform=train_tfms)
val_data = datasets.ImageFolder(r"C:\Users\sanjk\OneDrive\Desktop\rosedata", transform=val_tfms)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)


In [4]:
device = 'cpu'

model = timm.create_model('efficientnet_b3', pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 3)
model = model.to(device)


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [6]:
epochs = 15

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    val_loss = 0
    val_correct = 0
    val_total = 0
    model.eval()
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} "
          f"Train Acc: {100*correct/total:.2f}% "
          f"Val Loss: {val_loss/len(val_loader):.4f} "
          f"Val Acc: {100*val_correct/val_total:.2f}%")


Epoch [1/15] Train Loss: 0.9262 Train Acc: 42.69% Val Loss: 0.8139 Val Acc: 50.00%
Epoch [2/15] Train Loss: 0.7720 Train Acc: 48.34% Val Loss: 0.7443 Val Acc: 50.00%
Epoch [3/15] Train Loss: 0.7412 Train Acc: 44.19% Val Loss: 0.7254 Val Acc: 50.00%
Epoch [4/15] Train Loss: 0.7218 Train Acc: 46.84% Val Loss: 0.7213 Val Acc: 50.00%
Epoch [5/15] Train Loss: 0.7241 Train Acc: 44.02% Val Loss: 0.7112 Val Acc: 50.00%
Epoch [6/15] Train Loss: 0.7173 Train Acc: 48.17% Val Loss: 0.7091 Val Acc: 50.00%
Epoch [7/15] Train Loss: 0.7152 Train Acc: 47.18% Val Loss: 0.7095 Val Acc: 50.00%
Epoch [8/15] Train Loss: 0.7174 Train Acc: 43.19% Val Loss: 0.7056 Val Acc: 50.00%
Epoch [9/15] Train Loss: 0.7186 Train Acc: 43.69% Val Loss: 0.7059 Val Acc: 50.00%
Epoch [10/15] Train Loss: 0.7090 Train Acc: 46.01% Val Loss: 0.7023 Val Acc: 50.00%
Epoch [11/15] Train Loss: 0.7114 Train Acc: 45.02% Val Loss: 0.7020 Val Acc: 50.00%
Epoch [12/15] Train Loss: 0.7009 Train Acc: 50.17% Val Loss: 0.7015 Val Acc: 50.00%
E